# Part 5: Loss functions and optimizers

## Regression: mean squared error

In [1]:
import torch
from torch import nn

pred = torch.tensor([2.5, 0.0, 2.0])
target = torch.tensor([3.0, -0.5, 2.0])
print("by hand :", ((pred - target) ** 2).mean().item())
print("nn.MSELoss:", nn.MSELoss()(pred, target).item())

by hand : 0.1666666716337204
nn.MSELoss: 0.1666666716337204


## Binary classification: logits and BCEWithLogitsLoss

The network's last layer outputs a raw score called a **logit**. The sigmoid turns a logit into a probability. `BCEWithLogitsLoss` applies the sigmoid inside the loss, in a numerically safer way, so the network itself should not end in a sigmoid.

In [2]:
logits = torch.tensor([2.0, -1.0, 0.5])
labels = torch.tensor([1.0, 0.0, 1.0])
p = torch.sigmoid(logits)
print("probabilities:", p.round(decimals=3))
by_hand = -(labels * p.log() + (1 - labels) * (1 - p).log()).mean()
print("by hand            :", round(by_hand.item(), 4))
print("BCEWithLogitsLoss  :", round(nn.BCEWithLogitsLoss()(logits, labels).item(), 4))

probabilities: tensor([0.8810, 0.2690, 0.6220])
by hand            : 0.3048
BCEWithLogitsLoss  : 0.3048


## A confident wrong answer is expensive

In [3]:
bce = nn.BCEWithLogitsLoss()
one = torch.tensor([1.0])
for logit in (4.0, 0.0, -4.0):
    print(f"logit {logit:5.1f} (p = {torch.sigmoid(torch.tensor(logit)).item():.3f}) when the label is 1: loss {bce(torch.tensor([logit]), one).item():.3f}")

logit   4.0 (p = 0.982) when the label is 1: loss 0.018
logit   0.0 (p = 0.500) when the label is 1: loss 0.693
logit  -4.0 (p = 0.018) when the label is 1: loss 4.018


## Several classes: CrossEntropyLoss

The network outputs one logit per class, and the label is the class number as an integer (`int64`).

In [4]:
logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 0.2, 3.0]])
classes = torch.tensor([0, 2])
print("CrossEntropyLoss:", round(nn.CrossEntropyLoss()(logits, classes).item(), 4))

CrossEntropyLoss: 0.1755


## Optimizers: the learning rate matters

The same line-fit problem as Parts 1 and 3, with four different optimizer settings. Each run takes 100 steps from the same start.

In [5]:
import numpy as np

rng = np.random.default_rng(0)
x = torch.tensor(rng.uniform(0, 5, 50), dtype=torch.float32)
y = 3 * x + 2 + torch.tensor(rng.normal(0, 0.3, 50), dtype=torch.float32)


def fit(make_optimizer, steps=100):
    w = torch.zeros(1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    optimizer = make_optimizer([w, b])
    for _ in range(steps):
        loss = ((w * x + b - y) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss.item(), w.item(), b.item()


settings = {
    "SGD, lr 0.001": lambda ps: torch.optim.SGD(ps, lr=0.001),
    "SGD, lr 0.05": lambda ps: torch.optim.SGD(ps, lr=0.05),
    "SGD, lr 0.2": lambda ps: torch.optim.SGD(ps, lr=0.2),
    "Adam, lr 0.1": lambda ps: torch.optim.Adam(ps, lr=0.1),
}
for name, make in settings.items():
    loss, w, b = fit(make)
    print(f"{name:14s} final loss {loss:12.4f}  w {w:9.3f}  b {b:9.3f}")

SGD, lr 0.001  final loss       2.5707  w     2.847  b     0.891
SGD, lr 0.05   final loss       0.0922  w     3.063  b     1.821
SGD, lr 0.2    final loss          nan  w       nan  b       nan
Adam, lr 0.1   final loss       0.1778  w     2.834  b     2.511


## What optimizer.step() does

For plain SGD, `step()` is exactly the update you wrote by hand in Part 1: subtract the learning rate times the gradient.

In [6]:
w = torch.tensor([1.0], requires_grad=True)
opt = torch.optim.SGD([w], lr=0.1)
loss = (w * 3.0 - 6.0).pow(2).sum()
loss.backward()
print("gradient:", w.grad.item(), " (2 * (3 - 6) * 3)")
opt.step()
print("new w:", w.item(), " = 1.0 - 0.1 * (-18.0)")

gradient: -18.0  (2 * (3 - 6) * 3)
new w: 2.799999952316284  = 1.0 - 0.1 * (-18.0)


## Your turn

1. Try SGD with `lr` of 0.08, 0.1 and 0.12. Which one diverges, and what does the loss look like for the one that only just survives?
2. Run Adam with `lr` of 0.001, 0.1 and 1.0. Which ends with the lowest loss here? Would you trust `lr=1.0` on a real network, and why not?
3. Without running it, predict which of `logit = 6` and `logit = -6` costs more when the label is 1, then check.